In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models

(10000, 28, 28)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models

# fijamos la semilla de Python, NumPy y TensorFlow para que sea reproducible
tf.keras.utils.set_random_seed(42)

# cargamos MNIST que tiene 60.000 imágenes de entrenamiento y 10.000 de test
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# normalizamos los píxeles de 0-255 a 0-1 porque si no , el gradiente sale enorme y el paso η·∇L se dispara
# Con valores entre 0 y 1 el gradiente queda en una escala controlada
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# añadimos la dimensión del canal: (28, 28) -> (28, 28, 1)
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# definimos la arquitectura de la red
red_neuronas = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(8, (5, 5), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(24, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

# Descenso de gradiente: w = w - eta * gradiente, con eta = learning_rate
red_neuronas.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Entrenamos reservando un 10% del train para validar
red_neuronas.fit(x_train, y_train, epochs=15, batch_size=64, validation_split=0.1)

# Evaluamos en el conjunto de test
perdida, acierto = red_neuronas.evaluate(x_test, y_test)
print("Acierto en test:", acierto)

# Mostramos las capas y el número de parámetros
red_neuronas.summary()
print("Parámetros totales:", red_neuronas.count_params())

# Guardamos el red_neuronas para usarlo en el ataque FGSM
red_neuronas.save("red_mnist.keras")


Epoch 1/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9112 - loss: 0.2771 - val_accuracy: 0.9783 - val_loss: 0.0712
Epoch 2/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9776 - loss: 0.0735 - val_accuracy: 0.9870 - val_loss: 0.0514
Epoch 3/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.9839 - loss: 0.0524 - val_accuracy: 0.9885 - val_loss: 0.0449
Epoch 4/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9874 - loss: 0.0411 - val_accuracy: 0.9892 - val_loss: 0.0424
Epoch 5/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9901 - loss: 0.0339 - val_accuracy: 0.9898 - val_loss: 0.0401
Epoch 6/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9919 - loss: 0.0282 - val_accuracy: 0.9885 - val_loss: 0.0402
Epoch 7/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9933 - loss: 0.0234 - val_accuracy: 0.9890 - val_loss: 0.0422
Epoch 8/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9944 - loss: 0.0195 - val_accuracy: 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 8)      │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 24)     │         1,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 600)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        38,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,076 (160.46 KB)

 Trainable params: 41,074 (160.45 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Parámetros totales: 41074


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models

# fijamos la semilla de Python, NumPy y TensorFlow para que sea reproducible
tf.keras.utils.set_random_seed(42)

# cargamos MNIST que tiene 60.000 imágenes de entrenamiento y 10.000 de test
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# normalizamos los píxeles de 0-255 a 0-1 porque si no , el gradiente sale enorme y el paso η·∇L se dispara
# Con valores entre 0 y 1 el gradiente queda en una escala controlada
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# aplanamos las imágenes
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# definimos la arquitectura de la red
red_neuronas = models.Sequential([
    # Capa de entrada
    layers.Input(shape=(784,)),

    
    layers.Dense(356, activation='relu'),

    
    layers.Dense(128, activation='relu'),

    
    layers.Dense(64, activation='relu'),

    # Capa de salida
    layers.Dense(10, activation='softmax')
])

# Descenso de gradiente: w = w - eta * gradiente, con eta = learning_rate
red_neuronas.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Entrenamos reservando un 10% del train para validar
red_neuronas.fit(x_train, y_train, epochs=15, batch_size=64, validation_split=0.1)

# Evaluamos en el conjunto de test
perdida, acierto = red_neuronas.evaluate(x_test, y_test)
print("Acierto en test:", acierto)

# Mostramos las capas y el número de parámetros
red_neuronas.summary()
print("Parámetros totales:", red_neuronas.count_params())

# Guardamos el red_neuronas para usarlo en el ataque FGSM
red_neuronas.save("red_mnist.keras")


Epoch 1/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9126 - loss: 0.2856 - val_accuracy: 0.9692 - val_loss: 0.1028
Epoch 2/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9695 - loss: 0.1024 - val_accuracy: 0.9733 - val_loss: 0.0890
Epoch 3/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9808 - loss: 0.0650 - val_accuracy: 0.9770 - val_loss: 0.0766
Epoch 4/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9876 - loss: 0.0431 - val_accuracy: 0.9765 - val_loss: 0.0752
Epoch 5/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9912 - loss: 0.0306 - val_accuracy: 0.9772 - val_loss: 0.0823
Epoch 6/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9942 - loss: 0.0198 - val_accuracy: 0.9787 - val_loss: 0.0794
Epoch 7/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9946 - loss: 0.0176 - val_accuracy: 0.9797 - val_loss: 0.0791
Epoch 8/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9941 - loss: 0.0180 - val_accuracy: 0.

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 356)            │       279,460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │        45,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,064 (1.27 MB)

 Trainable params: 334,062 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Parámetros totales: 334062


Epoch 1/15


C:\Users\carlo\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


844/844 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6358 - loss: 1.3300 - val_accuracy: 0.8843 - val_loss: 0.4981
Epoch 2/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8863 - loss: 0.4302 - val_accuracy: 0.9228 - val_loss: 0.2880
Epoch 3/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9165 - loss: 0.3004 - val_accuracy: 0.9417 - val_loss: 0.2145
Epoch 4/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9341 - loss: 0.2320 - val_accuracy: 0.9553 - val_loss: 0.1709
Epoch 5/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9470 - loss: 0.1869 - val_accuracy: 0.9642 - val_loss: 0.1427
Epoch 6/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9556 - loss: 0.1559 - val_accuracy: 0.9677 - val_loss: 0.1251
Epoch 7/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9619 - loss: 0.1335 - val_accuracy: 0.9702 - val_loss: 0.1142
Epoch 8/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9673 - loss: 0.1166 - val_accuracy: 0.9712 - val_

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,148 (434.18 KB)

 Trainable params: 111,146 (434.16 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Parámetros totales: 111146
